# Naive Bayes Model for Fake News Detection

## Problem Overview

This notebook implements a Multinomial Naive Bayes classifier to distinguish between fake and real news articles. Naive Bayes is chosen for its simplicity, interpretability, and effectiveness on text classification tasks with high-dimensional sparse features (Bag-of-Words). The model uses frequency-based embeddings to capture word patterns that differentiate fake from real news content.

## Data Loading

Load the primary training datasets and apply initial cleaning steps. The Reuters attribution patterns are removed to prevent the model from learning dataset-specific artifacts rather than genuine linguistic patterns that distinguish fake from real news.

In [ ]:
import pandas as pd
import numpy as np

# Load primary training datasets
# These datasets will be used to train the Naive Bayes model
fake = pd.read_csv("../data/Fake.csv")
true = pd.read_csv("../data/True.csv")

# Load additional dataset for later evaluation (testing generalization)
new_fake_and_true = pd.read_csv("../data/Fake_Real_News_Data.csv")

# Add binary labels: 0 = Fake, 1 = Real
# This encoding is standard for binary classification and works well with Naive Bayes
fake["label"] = 0
true["label"] = 1

# Combine datasets into a single dataframe for processing
df = pd.concat([fake, true], ignore_index=True)

# Text Preprocessing: Remove dataset-specific artifacts
# Removing Reuters attribution patterns prevents the model from learning dataset-specific markers
# rather than genuine linguistic patterns that distinguish fake from real news
df['text'] = df['text'].str.replace(r'[\(\-–\s]*Reuters[\)\s]*', '', case=False, regex=True)
# Remove uppercase author prefixes (e.g., "JOHN SMITH - ") that may be dataset-specific
df['text'] = df['text'].str.replace(r'^[A-Z]+(?:\s+[A-Z]+)*-\s*', '', regex=True)

# Drop subject column - focusing on text content only for this model
# Subject could be a strong feature, but we want to evaluate text-based classification
df = df.drop(columns='subject')

df

,title,text,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...","December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...","December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,"December 25, 2017",0
...,...,...,...,...
44893,'Fully committed' NATO backs new U.S. approach...,NATO allies on Tuesday welcomed President Dona...,"August 22, 2017",1
44894,LexisNexis withdrew two products from Chinese ...,"LexisNexis, a provider of legal, regulatory an...","August 22, 2017",1
44895,Minsk cultural hub becomes haven from authorities,In the shadow of disused Soviet-era factories ...,"August 22, 2017",1
44896,Vatican upbeat on possibility of Pope Francis ...,Vatican Secretary of State Cardinal Pietro Par...,"August 22, 2017",1


In [ ]:
# Quick inspection of the additional evaluation dataset
# This dataset will be used later to test model generalization on unseen data
new_fake_and_true.head()
new_fake_and_true.tail()

,Unnamed: 0,title,text,label
6330,6330,Obama To Limit Police Acquisition Of Some Mili...,Obama To Limit Police Acquisition Of Some Mili...,REAL
6331,6331,EU using taxpayer money to give Muslim invader...,BNI Store Oct 29 2016 EU using taxpayer money ...,FAKE
6332,6332,Watching These 55 ISIS Terrorists Get Blown to...,Next Story → Judge Judy LOSES IT on Hood Rat: ...,FAKE
6333,6333,America’s Streets Will Run With Blood- Mike Adams,America’s Streets Will Run With Blood- Mike Ad...,FAKE
6334,6334,The immigration swamp,“This was not a subject that was on anybody’s ...,REAL


## Exploratory Data Analysis: Additional Dataset

Examining the structure and characteristics of the additional evaluation dataset to understand its format and ensure compatibility with our preprocessing pipeline.

In [ ]:
# Exploratory analysis of the evaluation dataset
# Checking dataset structure, size, and basic statistics
new_fake_and_true.head()
new_fake_and_true.tail()
new_fake_and_true.info()
new_fake_and_true.describe()
new_fake_and_true

# Note: Duplicate checking code is commented out but kept for reference
# This would help identify if evaluation data overlaps with training data (data leakage)
# duplicate = new_fake_and_true.merge(df, how='inner')
# print(f"Number of duplicate rows in new_df: {len(duplicate)}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  6335 non-null   int64 
 1   title       6335 non-null   object
 2   text        6335 non-null   object
 3   label       6335 non-null   object
dtypes: int64(1), object(3)
memory usage: 198.1+ KB


,Unnamed: 0,title,text,label
0,0,A whirlwind day in D.C. showcases Trump’s unor...,Donald Trump endorsed an unabashedly noninterv...,REAL
1,1,"In Baltimore's call for federal police probe, ...",While some Justice Department investigations a...,REAL
2,2,Trump Proudly Declares: Most Of The People I’v...,Trump Proudly Declares: Most Of The People I’v...,FAKE
3,3,Inside the Trump-Bush melodrama: Decades of te...,Donald Trump spent a day in January 2014 hobno...,REAL
4,4,Shutdown clash to return in force by December,Notable names include Ray Washburne (Commerce)...,REAL
...,...,...,...,...
6330,6330,Obama To Limit Police Acquisition Of Some Mili...,Obama To Limit Police Acquisition Of Some Mili...,REAL
6331,6331,EU using taxpayer money to give Muslim invader...,BNI Store Oct 29 2016 EU using taxpayer money ...,FAKE
6332,6332,Watching These 55 ISIS Terrorists Get Blown to...,Next Story → Judge Judy LOSES IT on Hood Rat: ...,FAKE
6333,6333,America’s Streets Will Run With Blood- Mike Adams,America’s Streets Will Run With Blood- Mike Ad...,FAKE


## Feature Engineering: Bag-of-Words Representation

Converting text into numerical features using Bag-of-Words (BoW). This approach counts word frequencies, creating a high-dimensional sparse matrix where each column represents a unique word in the vocabulary. BoW is well-suited for Naive Bayes because:
- It captures word frequency patterns that differentiate fake and real news
- Naive Bayes handles high-dimensional sparse data efficiently
- The independence assumption (though not strictly true) works reasonably well with word counts

## Model Architecture: Multinomial Naive Bayes

Multinomial Naive Bayes is chosen because:
- It models word counts (multinomial distribution) rather than binary presence/absence
- Efficiently handles high-dimensional sparse features from BoW
- Provides probabilistic predictions and interpretable feature importance
- Fast training and inference, making it suitable for large text datasets

The "naive" independence assumption (words are conditionally independent given class) is violated in practice, but Naive Bayes often performs well on text classification despite this limitation.

In [ ]:

from sklearn.feature_extraction.text import CountVectorizer

# Prepare text data: ensure all entries are strings (handle edge cases where text might be lists)
faketext = df[df['label'] == 0]['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
realtext = df[df['label'] == 1]['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

# Fit vectorizer on combined dataset to build shared vocabulary
# Critical: Using the same vocabulary ensures feature alignment between fake and real text
# This allows the model to compare word frequencies across classes meaningfully
vectorizer = CountVectorizer(stop_words='english')  # Remove common English stop words to reduce noise
vectorizer.fit(pd.concat([faketext, realtext]))

# Transform each class separately to get sparse word-count matrices
# Each row is a document, each column is a word in the vocabulary
X_fake_text = vectorizer.transform(faketext)
X_real_text = vectorizer.transform(realtext)

# Inspect vocabulary and sparse matrix structure
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())} unique words")
print(f"Sample vocabulary (first 20 words): {vectorizer.get_feature_names_out()[:20]}")

print(f"\nFake text matrix shape: {X_fake_text.shape}")
print(f"Real text matrix shape: {X_real_text.shape}")
print("\nSample sparse matrix representation (showing structure):")
print(X_fake_text)
print(X_real_text)

['00' '000' '0000' ... 'zzzzzzzz' 'zzzzzzzzzzzzz' 'émigré']
  (np.int32(0), np.int32(717))	1
  (np.int32(0), np.int32(2027))	1
  (np.int32(0), np.int32(2760))	1
  (np.int32(0), np.int32(3190))	1
  (np.int32(0), np.int32(3358))	2
  (np.int32(0), np.int32(3364))	1
  (np.int32(0), np.int32(3636))	1
  (np.int32(0), np.int32(3891))	1
  (np.int32(0), np.int32(4015))	2
  (np.int32(0), np.int32(4113))	2
  (np.int32(0), np.int32(4148))	1
  (np.int32(0), np.int32(4150))	1
  (np.int32(0), np.int32(4156))	3
  (np.int32(0), np.int32(5272))	13
  (np.int32(0), np.int32(5961))	1
  (np.int32(0), np.int32(6885))	1
  (np.int32(0), np.int32(8926))	1
  (np.int32(0), np.int32(10636))	1
  (np.int32(0), np.int32(10646))	1
  (np.int32(0), np.int32(11159))	1
  (np.int32(0), np.int32(11650))	3
  (np.int32(0), np.int32(11665))	1
  (np.int32(0), np.int32(11679))	1
  (np.int32(0), np.int32(12084))	1
  (np.int32(0), np.int32(12232))	1
  :	:
  (np.int32(23480), np.int32(111783))	1
  (np.int32(23480), np.int32(113114)

In [ ]:
## Training Procedure

from scipy.sparse import vstack
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Combine fake and real text matrices into a single feature matrix
# Using vstack for sparse matrices (memory efficient)
X_all = vstack([X_fake_text, X_real_text])
y_all = np.concatenate([
    np.zeros(len(faketext)),  # Label 0 for fake news
    np.ones(len(realtext))    # Label 1 for real news
])

# Train/test split: 80/20 split with random_state for reproducibility
# This split allows us to evaluate performance on held-out data from the same distribution
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

# Initialize and train Multinomial Naive Bayes
# Using default hyperparameters (alpha=1.0 for Laplace smoothing)
# Laplace smoothing prevents zero probabilities for unseen words
model = MultinomialNB()
model.fit(X_train, y_train)

# Evaluate on test set
y_pred = model.predict(X_test)
print("Test Set Accuracy:", accuracy_score(y_test, y_pred))
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

Multinomial Naive Bayes is chosen because:
- It models word counts (multinomial distribution) rather than binary presence/absence
- Efficiently handles high-dimensional sparse features from BoW
- Provides probabilistic predictions and interpretable feature importance
- Fast training and inference, making it suitable for large text datasets

The "naive" independence assumption (words are conditionally independent given class) is violated in practice, but Naive Bayes often performs well on text classification despite this limitation.

Accuracy: 0.9429844097995546
              precision    recall  f1-score   support

         0.0       0.94      0.95      0.95      4733
         1.0       0.94      0.93      0.94      4247

    accuracy                           0.94      8980
   macro avg       0.94      0.94      0.94      8980
weighted avg       0.94      0.94      0.94      8980



Testing Model on new dataset 1

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
new_fake_and_true = pd.read_csv("../data/Fake_Real_News_Data.csv")

if 'Unnamed: 0' in new_fake_and_true.columns:
    new_fake_and_true = new_fake_and_true.drop(columns='Unnamed: 0')

new_fake_and_true.head()

label_map = {'FAKE': 0, 'REAL': 1}
new_fake_and_true['label'] = new_fake_and_true['label'].map(label_map)

new_text = new_fake_and_true['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

X_new = vectorizer.transform(new_text)

y_new = new_fake_and_true['label'].values

y_new_pred = model.predict(X_new)

print(f"Accuracy score on new dataset:", accuracy_score(y_new, y_new_pred))
print(classification_report(y_new, y_new_pred, target_names =['Fake', 'Real']))




Accuracy score on new dataset: 0.582162588792423
              precision    recall  f1-score   support

        Fake       0.56      0.82      0.66      3164
        Real       0.66      0.34      0.45      3171

    accuracy                           0.58      6335
   macro avg       0.61      0.58      0.56      6335
weighted avg       0.61      0.58      0.56      6335



In [7]:
new_fake_and_true.head()

,title,text,label
0,A whirlwind day in D.C. showcases Trump’s unor...,Donald Trump endorsed an unabashedly noninterv...,1
1,"In Baltimore's call for federal police probe, ...",While some Justice Department investigations a...,1
2,Trump Proudly Declares: Most Of The People I’v...,Trump Proudly Declares: Most Of The People I’v...,0
3,Inside the Trump-Bush melodrama: Decades of te...,Donald Trump spent a day in January 2014 hobno...,1
4,Shutdown clash to return in force by December,Notable names include Ray Washburne (Commerce)...,1


In [ ]:
# Quick inspection of the evaluation dataset structure
new_fake_and_true.head()


Original dataset shape: (72134, 3)
Columns: ['title', 'text', 'label']

Missing values in 'text': 39
Missing values in 'title': 558

After filtering empty text: 71351 samples
Removed 783 empty text samples

Transforming text data using the trained vectorizer...
X_new shape: (71351, 122291)
Making predictions...

Accuracy score on WELFake dataset: 0.1930

Classification Report:
              precision    recall  f1-score   support

        Fake       0.24      0.29      0.26     35027
        Real       0.12      0.10      0.11     36324

    accuracy                           0.19     71351
   macro avg       0.18      0.19      0.19     71351
weighted avg       0.18      0.19      0.18     71351



## Additional Evaluation: WELFake Dataset

Testing on a third dataset (WELFake) to further assess generalization. This dataset is larger and may have different characteristics, providing additional insight into model robustness across diverse fake news sources.

# Testing Model on WELFake Dataset (with proper error handling)

# Load WELFake dataset
welfake_data = pd.read_csv("../data/WELFake_Dataset.csv")

if 'Unnamed: 0' in welfake_data.columns:
    welfake_data = welfake_data.drop(columns='Unnamed: 0')

print(f"Original dataset shape: {welfake_data.shape}")
print(f"Columns: {welfake_data.columns.tolist()}")

# Check for missing values - important for data quality assessment
print(f"\nMissing values in 'text': {welfake_data['text'].isna().sum()}")
print(f"Missing values in 'title': {welfake_data['title'].isna().sum()}")

# Handle label mapping - accommodate different label formats across datasets
if welfake_data['label'].dtype == 'object':
    label_map = {'FAKE': 0, 'REAL': 1, 'fake': 0, 'real': 1, 'Fake': 0, 'Real': 1}
    welfake_data['label'] = welfake_data['label'].map(label_map)

# Prepare text data - handle NaN and empty strings robustly
# Empty text would cause issues with vectorization
new_text = welfake_data['text'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else str(x) if pd.notna(x) else ''
)

# Filter out empty text before transforming
# Critical: Empty strings would result in zero-feature vectors, breaking predictions
non_empty_mask = new_text.str.strip().str.len() > 0
welfake_data_filtered = welfake_data[non_empty_mask].copy()
new_text_filtered = new_text[non_empty_mask]

print(f"\nAfter filtering empty text: {len(welfake_data_filtered)} samples")
print(f"Removed {len(welfake_data) - len(welfake_data_filtered)} empty text samples")

if len(new_text_filtered) == 0:
    raise ValueError("All text samples are empty after processing! Check your data cleaning steps.")

# Transform using the trained vectorizer (same vocabulary as training)
# Out-of-vocabulary words in WELFake will be ignored
print("\nTransforming text data using the trained vectorizer...")
X_new = vectorizer.transform(new_text_filtered)

print(f"X_new shape: {X_new.shape}")

if X_new.shape[0] == 0:
    raise ValueError("Vectorizer returned 0 samples. Check if text data is valid.")

# Get labels for filtered data
y_new = welfake_data_filtered['label'].values

# Make predictions
print("Making predictions...")
y_new_pred = model.predict(X_new)

# Evaluate performance
print(f"\nAccuracy score on WELFake dataset: {accuracy_score(y_new, y_new_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_new, y_new_pred, target_names=['Fake', 'Real']))

### Results & Observations: Severe Performance Degradation on WELFake Dataset

**Critical Finding:** Model accuracy drops dramatically to **19.3%** on the WELFake dataset, which is worse than random guessing (50% for binary classification). This indicates:

1. **Severe Distribution Mismatch**: The WELFake dataset likely has fundamentally different characteristics (writing style, vocabulary, topics, or labeling criteria) compared to the training data.

2. **Vocabulary Gap**: Many words in WELFake may not exist in the training vocabulary, leaving the model with insufficient signal to make accurate predictions.

3. **Possible Labeling Differences**: The definition of "fake" vs "real" may differ between datasets, or WELFake may have different quality/consistency in labels.

4. **Model Limitations**: This demonstrates that a model trained on one source of fake news may not generalize to others, highlighting the challenge of fake news detection across diverse sources.

**Note:** The model performs worse than a random classifier, suggesting it may be learning patterns that are counterproductive for this particular dataset distribution.

## Limitations & Next Steps

### Current Limitations

1. **Distribution Sensitivity**: The model shows poor generalization across different datasets, indicating overfitting to the training distribution.

2. **Vocabulary Dependency**: Performance degrades when evaluation data contains words not seen during training, limiting real-world applicability.

3. **Feature Representation**: Bag-of-Words loses word order and context, which may be important for detecting fake news patterns (e.g., misleading phrasing, logical inconsistencies).

4. **No Cross-Validation**: Single train/test split doesn't provide confidence intervals or robust performance estimates.

5. **Hyperparameter Tuning**: Default Naive Bayes parameters (alpha=1.0) may not be optimal; no hyperparameter search was performed.

### Potential Improvements

1. **Feature Engineering**:
   - Incorporate TF-IDF weighting to reduce importance of common words
   - Add n-gram features (bigrams, trigrams) to capture word order
   - Include metadata features (title, author, publication date if available)

2. **Model Improvements**:
   - Hyperparameter tuning (Laplace smoothing parameter alpha)
   - Ensemble methods combining multiple models
   - More sophisticated text representations (word embeddings, transformer-based features)

3. **Data Strategy**:
   - Combine multiple datasets during training to improve generalization
   - Domain adaptation techniques to handle distribution shift
   - Active learning to identify and label difficult cases

4. **Evaluation**:
   - Cross-validation for more robust performance estimates
   - Error analysis to understand failure modes
   - Confusion matrix analysis to identify systematic biases

5. **Robustness**:
   - Test on more diverse datasets
   - Adversarial testing to identify vulnerabilities
   - Confidence calibration to better assess prediction reliability

### Results & Observations: Performance Drop on New Dataset

**Key Finding:** Model accuracy drops from **94.3%** (test set) to **58.2%** (new dataset). This significant performance degradation indicates:

1. **Distribution Shift**: The new dataset likely comes from different sources or collection methods, with different writing styles, topics, or vocabulary patterns than the training data.

2. **Vocabulary Mismatch**: Words common in the new dataset may not exist in the training vocabulary, reducing the model's ability to make accurate predictions.

3. **Overfitting to Training Distribution**: The model learned patterns specific to the original datasets (Fake.csv/True.csv) that don't generalize well to other fake news sources.

**Class Imbalance Impact**: The model shows asymmetric performance - better recall on Fake (0.82) but poor recall on Real (0.34), suggesting it's more conservative in predicting Real news on this dataset.